# News Classification with GPT-2 Fine-Tuning

This project demonstrates how to fine-tune a GPT-2 model for news article classification using the AG News dataset. The workflow covers data preparation, model adaptation, training, evaluation, and inference, all implemented in PyTorch.

## Project Structure

All implementation details live in the `src/` package. This notebook handles orchestration, visualization, and inference examples.

- `src/config/` — Centralized configuration and logging
- `src/data/` — Dataset, dataloader, and preprocessing
- `src/models/` — GPT-2 model architecture
- `src/training/` — Training loop, evaluation, and metrics
- `src/inference/` — Inference pipeline
- `src/utils/` — Reproducibility seeding and GPT-2 weight loading helpers


## 1. Setup & Configuration


In [ ]:
import sys
import os

# Resolve project root whether Jupyter was started from the repo root or notebooks/
_cwd = os.getcwd()
_parent = os.path.abspath(os.path.join(_cwd, '..'))
if os.path.isdir(os.path.join(_cwd, 'src')):
    project_root = _cwd
elif os.path.isdir(os.path.join(_parent, 'src')):
    project_root = _parent
else:
    raise RuntimeError(
        'Could not find src/. Start Jupyter from the project root or notebooks/ directory.'
    )

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Change working directory to project root so relative paths
# (data/, models/, gpt2/, plots/) resolve correctly
os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

import torch
import tiktoken

from src.config.config import CONFIG, logger
from src.utils.seed import set_seed
from src.utils.helpers import download_and_load_gpt2, load_weights_into_gpt
from src.data.preprocessing import load_data
from src.data.dataset import NewsDataset
from src.data.dataloader import create_dataloaders
from src.models.gpt2_classifier import GPTModel, configure_transfer_learning
from src.training.train import train_classifier_simple
from src.training.metrics import (
    calc_accuracy_loader,
    generate_classification_metrics,
    save_accuracy_plot,
    save_loss_plot,
)
from src.inference.predict import classify_news


In [ ]:
# Set all random seeds for full reproducibility across torch, numpy, and Python
set_seed(CONFIG['training']['seed'])


## 2. Data Preparation


In [ ]:
# Download AG News (if needed) and load stratified train/val/test CSV splits
train_df, val_df, test_df = load_data()


## 3. Dataset & DataLoader


In [ ]:
tokenizer = tiktoken.get_encoding('gpt2')

# Per-sequence cap only (model context length). No global max across the split.
# custom_collate_fn pads each batch to the longest sequence in that batch only.
train_dataset = NewsDataset(data=train_df, tokenizer=tokenizer)
val_dataset = NewsDataset(data=val_df, tokenizer=tokenizer)
test_dataset = NewsDataset(data=test_df, tokenizer=tokenizer)

logger.info(f'Per-sequence token cap: {train_dataset.max_length} (context length)')
logger.info(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')


In [ ]:
# DataLoaders pad each batch to its longest sequence and build attention masks
train_loader, val_loader, test_loader = create_dataloaders(train_dataset, val_dataset, test_dataset)


## 4. Model Architecture & Pretrained Weights


In [ ]:
# Download and load pretrained GPT-2 weights
CHOOSE_MODEL = 'gpt2-small (124M)'
model_size = CHOOSE_MODEL.split(' ')[-1].lstrip('(').rstrip(')')
settings, params = download_and_load_gpt2(model_size=model_size, models_dir='gpt2')

model = GPTModel(CONFIG['gpt_model'])
load_weights_into_gpt(model, params)
model.eval()
logger.info('Pretrained GPT-2 weights loaded successfully.')


In [ ]:
set_seed(CONFIG['training']['seed'])
configure_transfer_learning(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
logger.info(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')


## 5. Pre-Training Baseline


In [ ]:
try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
except ImportError:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'Using device: {device}')
model.to(device)

set_seed(CONFIG['training']['seed'])

# Pre-training accuracy (subset of batches; size from CONFIG)
baseline_batches = CONFIG['training']['eval_iter']
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=baseline_batches)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=baseline_batches)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=baseline_batches)

print(f'Pre-training accuracy:')
print(f'  Train: {train_accuracy*100:.2f}%')
print(f'  Val:   {val_accuracy*100:.2f}%')
print(f'  Test:  {test_accuracy*100:.2f}%')


## 6. Training


In [ ]:
set_seed(CONFIG['training']['seed'])

# Only trainable parameters are passed to the optimizer
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['training']['lr'],
    weight_decay=CONFIG['training']['weight_decay']
)

eval_freq = max(1, len(train_loader) // CONFIG['training']['evals_per_epoch'])
logger.info(
    f"Evaluation frequency: every {eval_freq} steps "
    f"(~{CONFIG['training']['evals_per_epoch']} times per epoch)"
)

train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=CONFIG['training']['num_epochs'],
    eval_freq=eval_freq,
    eval_iter=CONFIG['training']['eval_iter'],
)


## 7. Load Best Checkpoint & Evaluate


In [ ]:
# Reload the best validation weights saved during training
# map_location and weights_only keep loading safe across devices
model_state_dict = torch.load(
    CONFIG['paths']['model_save_path'],
    map_location=device,
    weights_only=True
)
model.load_state_dict(model_state_dict)
logger.info('Best checkpoint loaded.')


## 8. Loss & Accuracy Plots


In [ ]:
save_loss_plot(train_losses, val_losses, examples_seen)
save_accuracy_plot(train_accs, val_accs)


In [ ]:
# Final accuracy on all splits (full evaluation)
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f'Final accuracy:')
print(f'  Train: {train_accuracy*100:.2f}%')
print(f'  Val:   {val_accuracy*100:.2f}%')
print(f'  Test:  {test_accuracy*100:.2f}%')


## 9. Classification Metrics


In [ ]:
# Confusion matrix and per-class classification report on the test set
generate_classification_metrics(test_loader, model, device)


## 10. Inference Examples


In [ ]:
text_1 = 'cricket lives in the heart and soul of every indian. stumps, bats and balls'
text_2 = 'ELON MUSK Looks Toward Commercial telephone market investment in reliance jio Group'
text_3 = 'The evolution of tesla cybertruck is amazing. the horse power of that car is amazing.'
text_4 = ('plane crash at ahmedabad airport causes shock to the nation. '
          'A bomb exploded during a Republic Day parade in India.')

for text in [text_1, text_2, text_3, text_4]:
    result = classify_news(text, model, tokenizer, device)
    print(f'{result:10s} | {text[:80]}...')
